# 04 — Merchant expansion opportunity

The score narrows a large district universe into an investigation queue. It does not forecast revenue or prove a merchant shortage.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
ANALYSIS = ROOT / "data" / "processed" / "analysis"
data = pd.read_csv(ANALYSIS / "district_opportunities.csv").sort_values("opportunity_rank")
validation = json.loads((ANALYSIS / "analysis_validation.json").read_text())
print(json.dumps(validation, indent=2))

{
  "latest_period": "2026-Q2",
  "district_metric_rows": 26616,
  "eligible_districts": 680,
  "score_in_0_100": true,
  "top10_positive_yoy_all_last_4_quarters": true,
  "equal_weight_top10_overlap": 9,
  "no_intensity_top10_overlap": 6
}


## Method

Eligibility requires at least 100,000 registered users, 1,000 registered merchants, one million quarterly transactions, and sufficient history for QoQ and same-quarter YoY growth. Percentile ranks limit outlier influence. The default score weights are 30% YoY growth, 25% transactions per merchant, 25% users per merchant, and 20% user base. Segments use observed score and scale quartiles: EXPAND, DEFEND, DEVELOP, and MONITOR.

In [2]:
top10 = data.head(10)[
    [
        "opportunity_rank",
        "district",
        "state",
        "opportunity_score",
        "transaction_yoy",
        "users_per_merchant",
        "positive_yoy_quarters_last4",
        "business_segment",
        "no_intensity_rank",
    ]
].copy()
top10["opportunity_score"] = top10["opportunity_score"].round(1)
top10["transaction_yoy"] = top10["transaction_yoy"].map(lambda x: f"{x:.1%}")
display(top10)
display(data["business_segment"].value_counts().rename_axis("segment").to_frame("districts"))

,opportunity_rank,district,state,opportunity_score,transaction_yoy,users_per_merchant,positive_yoy_quarters_last4,business_segment,no_intensity_rank
1,1,west godavari,andhra pradesh,86.7,30.5%,24.61701,4.0,EXPAND,2
3,2,sangareddy,telangana,86.3,32.3%,18.01938,4.0,EXPAND,4
5,3,dr br ambedkar konaseema,andhra pradesh,85.0,33.0%,20.38370,4.0,EXPAND,6
6,4,east godavari,andhra pradesh,84.5,32.2%,18.27581,4.0,EXPAND,7
8,5,sri potti sriramulu nellore,andhra pradesh,84.1,30.5%,18.48564,4.0,EXPAND,9
7,6,alluri sitharama raju,andhra pradesh,83.6,37.9%,36.60389,4.0,EXPAND,8
15,7,tirupati,andhra pradesh,82.3,28.2%,18.66422,4.0,EXPAND,16
20,8,eluru,andhra pradesh,81.4,29.2%,18.91242,4.0,EXPAND,21
19,9,mandya,karnataka,81.1,28.2%,23.76374,4.0,EXPAND,20
22,10,hassan,karnataka,80.3,26.4%,26.22027,4.0,EXPAND,23


,districts
segment,
MONITOR,273
DEVELOP,154
EXPAND,132
DEFEND,121


In [3]:
base = set(data.nsmallest(10, "opportunity_rank")["district_key"])
equal = set(data.nsmallest(10, "equal_weight_rank")["district_key"])
without_intensity = set(data.nsmallest(10, "no_intensity_rank")["district_key"])
print("Equal-weight top-10 overlap:", len(base & equal))
print("No-intensity top-10 overlap:", len(base & without_intensity))
assert data["opportunity_score"].between(0, 100).all()
assert data.head(10)["positive_yoy_quarters_last4"].eq(4).all()

Equal-weight top-10 overlap: 9
No-intensity top-10 overlap: 6


## Recommendation

Begin field validation with West Godavari and Sangareddy. The stronger follow-on group is Dr BR Ambedkar Konaseema, East Godavari, Sri Potti Sriramulu Nellore, and Alluri Sitharama Raju because all six remain in the no-intensity top ten. Every default top-ten district has positive YoY demand growth in all four latest quarters. Validate active acceptance, category gaps, competitor coverage, acquisition cost, and travel effort before committing budget.